In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Multiply, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer

print("1. Locating Dataset in Kaggle Environment...")
kaggle_input_dir = '/kaggle/input/'
csv_paths = glob.glob(os.path.join(kaggle_input_dir, '**/*.csv'), recursive=True)
image_paths = glob.glob(os.path.join(kaggle_input_dir, '**/*150x150/**/*.jpg'), recursive=True)

csv_path = csv_paths[0] if csv_paths else None
print(f"   CSV Found at: {csv_path}")
print(f"   Total Images Found: {len(image_paths)}")

print("\n2. Building Data Mapping...")
df_labels = pd.read_csv(csv_path, low_memory=False)

filename_to_path = {os.path.basename(p): p for p in image_paths}
df_labels['filename'] = df_labels['image_url'].apply(lambda x: str(x).split('/')[-1])
df_labels['filepath'] = df_labels['filename'].map(filename_to_path)

df_clean = df_labels.dropna(subset=['filepath']).copy()

weather_map = {'S': 'SUNNY', 'O': 'OVERCAST', 'R': 'RAINY'}
df_clean['weather_text'] = df_clean['weather'].map(weather_map)
df_clean['label'] = df_clean['occupancy'].astype(int)

lb = LabelBinarizer()
weather_encoded = lb.fit_transform(df_clean['weather_text'])
df_clean['weather_0'] = weather_encoded[:, 0]
df_clean['weather_1'] = weather_encoded[:, 1]
df_clean['weather_2'] = weather_encoded[:, 2]

print(f"   Successfully prepared {len(df_clean)} images for training!")

print("\n3. Splitting Data & Creating Pipeline...")
train_df, val_df = train_test_split(df_clean, test_size=0.2, random_state=42)

def process_data(filepath, weather_vec, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [150, 150])
    img = img / 255.0
    return (img, weather_vec), label

def create_dataset(dataframe, batch_size=64): 
    filepaths = dataframe['filepath'].values
    weather_data = dataframe[['weather_0', 'weather_1', 'weather_2']].values.astype('float32')
    labels = dataframe['label'].values.astype('float32')
    
    dataset = tf.data.Dataset.from_tensor_slices((filepaths, weather_data, labels))
    dataset = dataset.map(process_data, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

BATCH_SIZE = 64
train_dataset = create_dataset(train_df, BATCH_SIZE)
val_dataset = create_dataset(val_df, BATCH_SIZE)

print("\n4. Building Condition-Gated CNN Architecture...")
image_input = Input(shape=(150, 150, 3), name='image_input')
x = Conv2D(32, (3, 3), activation='relu')(image_input)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(64, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = BatchNormalization()(x)
x = Conv2D(128, (3, 3), activation='relu')(x)
x = MaxPooling2D((2, 2))(x)
x = Flatten()(x)
image_features = Dense(64, activation='relu')(x)

weather_input = Input(shape=(3,), name='weather_input')
w = Dense(16, activation='relu')(weather_input)
weather_gate = Dense(64, activation='sigmoid', name='weather_gate')(w)

gated_features = Multiply(name='Condition_Gating')([image_features, weather_gate])

out = Dense(32, activation='relu')(gated_features)
out = Dropout(0.5)(out)
final_output = Dense(1, activation='sigmoid', name='occupancy_output')(out)

model = Model(inputs=[image_input, weather_input], outputs=final_output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n5. Starting Full GPU Training...")
model_save_path = '/kaggle/working/best_parking_model.keras'

callbacks = [
    ModelCheckpoint(model_save_path, save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1)
]

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=callbacks
)

print(f"\nTraining Completed! Best model saved at: {model_save_path}")

2026-03-05 10:06:55.157700: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772705215.387281      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772705215.447818      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772705215.977461      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772705215.977503      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772705215.977505      55 computation_placer.cc:177] computation placer alr

1. Locating Dataset in Kaggle Environment...
   CSV Found at: /kaggle/input/datasets/ddsshubham/cnrpark-ext/CNRParkEXT.csv
   Total Images Found: 157549

2. Building Data Mapping...
   Successfully prepared 157549 images for training!

3. Splitting Data & Creating Pipeline...


I0000 00:00:1772705658.092718      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772705658.098629      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



4. Building Condition-Gated CNN Architecture...

5. Starting Full GPU Training...
Epoch 1/15


I0000 00:00:1772705662.082151     129 service.cc:152] XLA service 0x7a127c011d80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772705662.082189     129 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1772705662.082194     129 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1772705662.730495     129 cuda_dnn.cc:529] Loaded cuDNN version 91002


   3/1970 ━━━━━━━━━━━━━━━━━━━━ 1:55 59ms/step - accuracy: 0.5981 - loss: 1.1839

I0000 00:00:1772705668.238314     129 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1970/1970 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9406 - loss: 0.1595
Epoch 1: val_accuracy improved from -inf to 0.68258, saving model to /kaggle/working/best_parking_model.keras
1970/1970 ━━━━━━━━━━━━━━━━━━━━ 125s 59ms/step - accuracy: 0.9406 - loss: 0.1594 - val_accuracy: 0.6826 - val_loss: 0.8785
Epoch 2/15
1969/1970 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9795 - loss: 0.0644
Epoch 2: val_accuracy improved from 0.68258 to 0.92736, saving model to /kaggle/working/best_parking_model.keras
1970/1970 ━━━━━━━━━━━━━━━━━━━━ 89s 45ms/step - accuracy: 0.9796 - loss: 0.0644 - val_accuracy: 0.9274 - val_loss: 0.1930
Epoch 3/15
1969/1970 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.9868 - loss: 0.0411
Epoch 3: val_accuracy improved from 0.92736 to 0.98997, saving model to /kaggle/working/best_parking_model.keras
1970/1970 ━━━━━━━━━━━━━━━━━━━━ 91s 46ms/step - accuracy: 0.9868 - loss: 0.0411 - val_accuracy: 0.9900 - val_loss: 0.0325
Epoch 4/15
1969/1970 ━━━━━━━━━━━━━━━━━━━━ 0s